# Module 7: AI in Banking and Finance - Use Case & Prompt Studio

This notebook supports the hands-on activities for Module 7 of the AJB AI and Data Training Programme.

You will:
1. Load and explore a dataset of banking AI use cases
2. Analyse use cases by risk, department, and AI type
3. Work with prompt templates and evaluate their structure
4. Build a prioritisation framework for AI adoption

**Prerequisites:** Modules 1-6 of the programme (Python fundamentals, data analysis, visualisation, ML concepts, NLP basics, dashboards).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["font.size"] = 11

In [ ]:
# Load the AI use cases dataset
use_cases = pd.read_csv("../data/ai_use_cases_banking.csv")
print(f"Use cases loaded: {len(use_cases)} rows, {len(use_cases.columns)} columns")
print(f"Columns: {list(use_cases.columns)}")
use_cases.head(10)

In [ ]:
# --- Exercise 1: Use Case Landscape Analysis ---
#
# Task A: Count use cases by department and display as a bar chart.

dept_counts = use_cases["department"].value_counts()
print("Use cases by department:")
print(dept_counts)
print()

dept_counts.plot(kind="barh", color="#2c5f8a", edgecolor="white")
plt.title("AI Use Cases by Department")
plt.xlabel("Number of Use Cases")
plt.ylabel("Department")
plt.tight_layout()
plt.show()

# Task B: Compare GenAI vs Traditional ML use cases.

ai_type_counts = use_cases["ai_type"].value_counts()
print("\nAI type distribution:")
print(ai_type_counts)

# Task C: Which departments rely more on GenAI vs Traditional ML?

cross_tab = pd.crosstab(use_cases["department"], use_cases["ai_type"])
print("\nDepartment x AI Type:")
print(cross_tab)

cross_tab.plot(kind="barh", stacked=True, color=["#2c5f8a", "#e8a838"], edgecolor="white")
plt.title("AI Type by Department")
plt.xlabel("Number of Use Cases")
plt.tight_layout()
plt.show()

In [ ]:
# --- Exercise 2: Risk and Compliance Assessment ---
#
# Task A: Create a risk-compliance matrix showing how many use cases
# fall into each combination of risk_rating and compliance_impact.

risk_compliance = pd.crosstab(use_cases["risk_rating"], use_cases["compliance_impact"])
print("Risk Rating vs Compliance Impact:")
print(risk_compliance)
print()

# Task B: Identify high-risk, high-compliance use cases that need
# the strongest oversight controls.

high_risk = use_cases[
    (use_cases["risk_rating"] == "High") & (use_cases["compliance_impact"] == "High")
]
print(f"High-risk, high-compliance use cases ({len(high_risk)}):")
print(high_risk[["use_case_id", "department", "use_case_name", "current_status"]].to_string(index=False))
print()

# Task C: Calculate total estimated value by risk rating.

value_by_risk = use_cases.groupby("risk_rating")["estimated_value_sar"].agg(["sum", "mean", "count"])
value_by_risk.columns = ["Total Value (SAR)", "Avg Value (SAR)", "Count"]
print("Value by risk rating:")
print(value_by_risk)

# Task D (Stretch): What percentage of high-risk use cases are still
# in evaluation? What does that tell you about adoption readiness?

high_risk_status = high_risk["current_status"].value_counts(normalize=True) * 100
print("\nHigh-risk use case status distribution (%):")
print(high_risk_status.round(1))

In [ ]:
# --- Exercise 3: Prompt Template Analysis ---
#
# Load and explore the prompt templates dataset.

prompts = pd.read_csv("../data/prompt_templates.csv")
print(f"Prompt templates loaded: {len(prompts)} rows")
prompts.head()


In [ ]:
# Task A: Examine the relationship between prompt risk level and
# constraint complexity (measured by constraint text length).

prompts["constraint_length"] = prompts["constraints"].str.len()
prompts["constraint_count"] = prompts["constraints"].str.count(";")

risk_constraint = prompts.groupby("risk_level").agg(
    avg_constraint_length=("constraint_length", "mean"),
    avg_constraint_count=("constraint_count", "mean"),
    template_count=("template_id", "count")
).round(1)

print("Constraint complexity by risk level:")
print(risk_constraint)
print()

# Task B: For each prompt pattern, list the constraints.
# Identify which patterns enforce the most guardrails.

print("Prompt patterns and their constraints:")
for _, row in prompts.iterrows():
    print(f"\n  {row['template_id']} | {row['prompt_pattern']} | Risk: {row['risk_level']}")
    print(f"    Constraints: {row['constraints']}")

# Task C (Stretch): Pick one high-risk prompt template.
# Write an improved version with additional banking-specific constraints.
# Add your improved prompt as a new row below.

print("\n--- Your improved prompt template ---")
print("Template ID: PT-???")
print("Use case: [your choice]")
print("Pattern: [your pattern]")
print("Constraints: [your constraints]")
print("Expected output: [your output type]")
print("Risk level: [your assessment]")

In [ ]:
# --- Exercise 4: AI Adoption Prioritisation ---
#
# Build a simple scoring model to prioritise use cases for AI adoption.
# This synthesises concepts from the full programme: data analysis
# (Modules 2-3), scoring logic (Module 4), and strategic thinking
# (this module).

# Define scoring weights
risk_score = {"Low": 3, "Medium": 2, "High": 1}
complexity_score = {"Low": 3, "Medium": 2, "High": 1}
status_score = {"Production": 4, "Pilot": 3, "Evaluation": 2}

scored = use_cases.copy()
scored["risk_points"] = scored["risk_rating"].map(risk_score)
scored["complexity_points"] = scored["implementation_complexity"].map(complexity_score)
scored["status_points"] = scored["current_status"].map(status_score)

# Normalise value to 1-3 scale
max_val = scored["estimated_value_sar"].max()
scored["value_points"] = (scored["estimated_value_sar"] / max_val * 3).round(1)

# Composite score (equal weights for this exercise)
scored["priority_score"] = (
    scored["risk_points"]
    + scored["complexity_points"]
    + scored["status_points"]
    + scored["value_points"]
).round(1)

# Show prioritised list
priority_view = scored[[
    "use_case_id", "department", "use_case_name",
    "risk_points", "complexity_points", "status_points",
    "value_points", "priority_score"
]].sort_values("priority_score", ascending=False)

print("AI Use Case Priority Ranking:")
print(priority_view.to_string(index=False))

# Visualise
fig, ax = plt.subplots(figsize=(12, 6))
colors = ["#2c5f8a" if s >= 10 else "#e8a838" if s >= 8 else "#cccccc" for s in priority_view["priority_score"]]
ax.barh(priority_view["use_case_name"], priority_view["priority_score"], color=colors, edgecolor="white")
ax.set_xlabel("Priority Score")
ax.set_title("AI Use Case Prioritisation - AJB")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

print("\nTop 5 priority use cases for next quarter:")
print(priority_view.head(5)[["use_case_id", "use_case_name", "priority_score"]].to_string(index=False))

In [ ]:
# --- Reflection ---
#
# Answer these questions in the markdown cell below:
#
# 1. Which use case would you recommend AJB prioritise next and why?
#    Consider risk, value, and current readiness.
#
# 2. What additional data would improve this prioritisation model?
#    Think about what is missing from the dataset.
#
# 3. How would you adjust the scoring weights for a bank that is
#    more risk-averse vs one that is more innovation-focused?
#
# 4. Looking back at the full programme (Modules 1-7), which skills
#    from earlier modules were most useful for this analysis?

print("Complete your reflection in the cell below.")

## Your Reflection

**1. Recommended use case and rationale:**

[Your answer here]

**2. Additional data needed:**

[Your answer here]

**3. Scoring weight adjustments:**

[Your answer here]

**4. Programme skills synthesis:**

[Your answer here]